# DEPENDENCY INSTALLATION

In [ ]:
!pip install -q "transformers>=4.35.0" "accelerate>=0.24.0"
!pip install -q torch torchaudio sentencepiece protobuf pandas
!pip install -q faster-whisper ctranslate2 huggingface_hub librosa

print("✅ Dependencies installed.")

# IMPORTS

In [ ]:
import gc
import os
import re
import time
import unicodedata
import warnings
import threading
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Tuple
from zoneinfo import ZoneInfo

import ctranslate2
import librosa
import numpy as np
import pandas as pd
import torch
import torchaudio
import transformers
from huggingface_hub import login
from tqdm.auto import tqdm
from transformers import WhisperForConditionalGeneration, WhisperProcessor

warnings.filterwarnings("ignore")
print("✅ Imports ready.")

# CONFIGURATION
- All runtime parameters are defined here. Edit before running.

In [ ]:
# ── HuggingFace Model ─────────────────────────────────────────────────────────
HF_MODEL_ID       = "bitwisemind/sam_15000_clean_text_full_model"
HF_TOKEN          = "Api_Key"          # paste your token here if the model is private

# ── CTranslate2 Model ─────────────────────────────────────────────────────────
CT2_MODEL_DIR     = "whisper-tiny-ct2"   # local output dir of ct2-transformers-converter

# ── Paths ─────────────────────────────────────────────────────────────────────
AUDIO_BASE_PATH   = (
    "/kaggle/input/competitions/"
    "dl-sprint-4-0-bengali-long-form-speech-recognition/"
    "transcription/transcription/test/audio"
)
OUTPUT_CSV        = "/kaggle/working/transcriptions_ct2.csv"

# ── Chunking & Batching ───────────────────────────────────────────────────────
CHUNK_S           = 25    # audio chunk length in seconds
OVERLAP_S         = 0     # overlap between chunks in seconds
BATCH_SIZE        = 8     # chunks per GPU per batch

# ── Multi-GPU ─────────────────────────────────────────────────────────────────
NUM_GPUS          = 2     # number of CUDA GPUs to use

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE            = "cuda" if torch.cuda.is_available() else "cpu"

print("✅ Configuration loaded.")
print(f"   HF Model      : {HF_MODEL_ID}")
print(f"   CT2 Model Dir : {CT2_MODEL_DIR}")
print(f"   Device        : {DEVICE}")
print(f"   GPUs          : {NUM_GPUS}  |  Chunk: {CHUNK_S}s  |  Batch: {BATCH_SIZE}")
print(f"   Output CSV    : {OUTPUT_CSV}")

# AUDIO UTILITIES

In [ ]:
def format_time(seconds: float) -> str:
    """Convert a duration in seconds to a human-readable h/m/s string."""
    h, rem = divmod(int(seconds), 3600)
    m, s   = divmod(rem, 60)
    if h > 0: return f"{h}h {m}m {s}s"
    if m > 0: return f"{m}m {s}s"
    return f"{s}s"


def load_audio(path: str) -> np.ndarray:
    """Load an audio file as a mono 16 kHz float32 numpy array."""
    audio, _ = librosa.load(path, sr=16000, mono=True)
    return audio


def make_chunks(audio: np.ndarray, chunk_s: int = CHUNK_S,
                overlap_s: int = OVERLAP_S) -> List[np.ndarray]:
    """Split audio into fixed-length chunks; pad the last one if needed."""
    chunk_size    = chunk_s   * 16000
    overlap_size  = overlap_s * 16000
    stride        = chunk_size - overlap_size
    chunks, start = [], 0

    while start < len(audio):
        end   = min(start + chunk_size, len(audio))
        chunk = audio[start:end]
        if len(chunk) < chunk_size:
            chunk = np.pad(chunk, (0, chunk_size - len(chunk)))
        chunks.append(chunk)
        if end >= len(audio):
            break
        start += stride

    return chunks


def merge_transcriptions(transcriptions: List[str]) -> str:
    """Join per-chunk transcriptions into a single string, collapsing whitespace."""
    if not transcriptions:
        return ""
    return " ".join(" ".join(transcriptions).split())


def validate_audio_files(
    audio_files: List[str], base_path: str
) -> Tuple[List[Path], List[str]]:
    """Check which requested audio files exist on disk."""
    base = Path(base_path)
    valid, missing = [], []
    for f in audio_files:
        p = base / f
        (valid if p.exists() else missing).append(p if p.exists() else f)
    return valid, missing


print("✅ Audio utilities ready.")

# TEXT NORMALIZATION & POST-PROCESSING

In [ ]:
# Zero-width Unicode characters frequently introduced by Bengali ASR outputs
_ZW_PATTERN = re.compile(r"[\u200B-\u200D\uFEFF]")


def normalize_bn_text(text: str) -> str:
    """Normalize Bengali text: NFC unicode, remove zero-width chars, collapse spaces."""
    if not text:
        return ""
    text = unicodedata.normalize("NFC", str(text))
    text = _ZW_PATTERN.sub("", text)
    text = text.replace("\u00A0", " ")   # non-breaking space → regular space
    return " ".join(text.split())


def clean_transcript(text: str) -> str:
    """Remove ASR hallucination artifacts from a transcript.

    Handles three hallucination patterns common in Whisper outputs:
      1. Multi-word phrases repeated 3+ times in sequence.
      2. Single words repeated 3+ times in sequence.
      3. Character n-grams repeated without spaces (e.g., "হেহেহেহে").
    Also strips '>>' speaker-change markers.
    """
    if not text or (isinstance(text, float) and pd.isna(text)):
        return text

    text = str(text).replace(">>", "")

    for pattern, replacement in [
        (r"\b((?:\S+\s+){1,5}\S+)(?:\s+\1){2,}\b", r"\1"),   # repeated phrases
        (r"\b(\S+)(?:\s+\1){2,}\b",                 r"\1"),   # repeated words
        (r"(.{2,}?)\1{2,}",                          r"\1"),   # repeated char-grams
    ]:
        prev = None
        while prev != text:
            prev = text
            text = re.sub(pattern, replacement, text)

    return re.sub(r"\s+", " ", text).strip()


def normalize_and_postprocess(text: str) -> str:
    """Apply Unicode normalization then hallucination-removal to a transcript."""
   #return clean_transcript(normalize_bn_text(text))
    return normalize_bn_text(text)


print("✅ Text normalization utilities ready.")

# MULTI-GPU CTRANSLATE2 INFERENCE (2× GPU)
- **Step 1** — Authenticate with HuggingFace (private models only).
- **Step 2** — Convert HuggingFace model to CTranslate2 format (run once).
- **Step 3** — Load processor, CT2 models, and Bengali prompt.
- **Step 4** — Define inference engine functions.
- **Step 5** — Run inference; normalization + post-processing applied per-chunk and on final merged output.

In [ ]:
# Authenticate with HuggingFace (required only for private models)
if HF_TOKEN:
    login(token=HF_TOKEN)
    print("✅ Logged in to HuggingFace Hub.")
else:
    print("ℹ️  HF_TOKEN not set — skipping login (public models only).")

In [ ]:
# Convert HuggingFace Whisper → CTranslate2 format (run once; skip if already done)
import os
if not os.path.isdir(CT2_MODEL_DIR):
    print(f"🔄 Converting {HF_MODEL_ID} → {CT2_MODEL_DIR} ...")
    os.system(
        f"ct2-transformers-converter "
        f"--model {HF_MODEL_ID} "
        f"--output_dir {CT2_MODEL_DIR} "
        f"--quantization float16"
    )
    print(f"✅ Conversion complete → {CT2_MODEL_DIR}")
else:
    print(f"✅ CT2 model already exists at '{CT2_MODEL_DIR}' — skipping conversion.")

In [ ]:
torch.cuda.empty_cache(); gc.collect()

# ── Processor ─────────────────────────────────────────────────────────────────
print(f"📥 Loading WhisperProcessor : {HF_MODEL_ID}")
ct2_processor = WhisperProcessor.from_pretrained(HF_MODEL_ID)

# ── CTranslate2 models — one per GPU ──────────────────────────────────────────
print(f"📥 Loading CTranslate2 model on {NUM_GPUS} GPU(s) ...")
ct2_models = [
    ctranslate2.models.Whisper(
        CT2_MODEL_DIR,
        device       = "cuda",
        device_index = gpu_id,
        compute_type = "float16",
        inter_threads = 2,
        intra_threads = 4,
    )
    for gpu_id in range(NUM_GPUS)
]

# ── Bengali forced-decoder prompt ─────────────────────────────────────────────
BN_PROMPT = ct2_processor.tokenizer.convert_tokens_to_ids([
    "<|startoftranscript|>",
    "<|bn|>",
    "<|transcribe|>",
    "<|notimestamps|>",
])

print(f"✅ CT2 models ready on {NUM_GPUS} GPU(s).")
print(f"   Processor : {HF_MODEL_ID}")
print(f"   Prompt    : {BN_PROMPT}")

In [ ]:
# ── Per-GPU worker ────────────────────────────────────────────────────────────
def process_chunks(
    gpu_id      : int,
    chunk_indices: List[int],
    chunks      : List[np.ndarray],
    results_dict: dict,
    lock        : threading.Lock,
) -> None:
    """
    Transcribe a subset of audio chunks on one GPU.
    Normalization + post-processing is applied to every decoded chunk
    before the result is stored.
    """
    model = ct2_models[gpu_id]

    for i in range(0, len(chunk_indices), BATCH_SIZE):
        batch_indices = chunk_indices[i : i + BATCH_SIZE]
        batch         = [chunks[j] for j in batch_indices]

        inputs   = ct2_processor(batch, return_tensors="np", sampling_rate=16000)
        features = ctranslate2.StorageView.from_array(inputs.input_features)
        outputs  = model.generate(features, [BN_PROMPT] * len(batch))

        for j, result in zip(batch_indices, outputs):
            raw_text   = ct2_processor.decode(
                result.sequences_ids[0], skip_special_tokens=True
            )
            clean_text = normalize_and_postprocess(raw_text)   # ← per-chunk
            with lock:
                results_dict[j] = clean_text
            print(f"[GPU {gpu_id}] Chunk {j + 1}/{len(chunks)}: {clean_text[:80]}...")


# ── Single-file pipeline ──────────────────────────────────────────────────────
def transcribe_file_ct2(audio_path: str) -> Dict:
    """
    End-to-end CTranslate2 pipeline for one audio file:
      load → chunk → parallel GPU inference (per-chunk normalisation)
      → ordered merge → final normalize & post-process → result dict.
    """
    name       = Path(audio_path).name
    start_time = time.time()

    audio      = load_audio(audio_path)
    chunks     = make_chunks(audio)
    duration_s = len(audio) / 16000
    print(f"   {name}  |  {format_time(duration_s)}  |  {len(chunks)} chunks")

    # Distribute chunk indices round-robin across GPUs
    all_indices = list(range(len(chunks)))
    gpu_indices = [all_indices[g::NUM_GPUS] for g in range(NUM_GPUS)]

    results_dict : Dict           = {}
    lock         : threading.Lock = threading.Lock()

    # Run all GPUs in parallel
    with ThreadPoolExecutor(max_workers=NUM_GPUS) as executor:
        futures = [
            executor.submit(
                process_chunks, gpu_id, gpu_indices[gpu_id],
                chunks, results_dict, lock
            )
            for gpu_id in range(NUM_GPUS)
        ]
        for f in futures:
            f.result()

    # Merge in original order, then apply final post-processing pass
    ordered_texts    = [results_dict[i] for i in range(len(chunks))]
    merged           = merge_transcriptions(ordered_texts)
    final_transcript = normalize_and_postprocess(merged)          # ← final pass

    elapsed = time.time() - start_time
    print(
        f"   ✓ Done in {format_time(elapsed)}  |  "
        f"speed {duration_s / elapsed:.2f}× realtime  |  "
        f"{len(final_transcript)} chars"
    )
    return {
        "filename"          : Path(audio_path).stem,
        "transcript"        : final_transcript,
        "duration_s"        : round(duration_s, 2),
        "num_chunks"        : len(chunks),
        "processing_time_s" : round(elapsed, 2),
        "audio_path"        : audio_path,
    }


# ── Multi-file runner ─────────────────────────────────────────────────────────
def run_ct2_inference(base_path: str, output_csv: str) -> pd.DataFrame:
    """
    Auto-discover all audio files in base_path, transcribe each one,
    write filename + transcript to CSV, and return the results DataFrame.
    """
    audio_paths = sorted(
        p for p in Path(base_path).iterdir()
        if p.suffix.lower() in (".wav", ".mp3", ".flac", ".ogg", ".m4a")
    )
    if not audio_paths:
        print("❌ No audio files found in:", base_path)
        return pd.DataFrame()

    print(f"   Found {len(audio_paths)} audio file(s) to process.")
    results, total_start = [], time.time()

    for idx, path in enumerate(audio_paths, 1):
        print(f"\n[{idx}/{len(audio_paths)}]")
        try:
            results.append(transcribe_file_ct2(str(path)))
        except Exception as exc:
            import traceback
            print(f"   ❌ Failed: {exc}")
            traceback.print_exc()
            results.append({
                "filename"          : path.stem,
                "transcript"        : f"[ERROR: {exc}]",
                "duration_s"        : 0,
                "num_chunks"        : 0,
                "processing_time_s" : 0,
                "audio_path"        : str(path),
            })

    df = pd.DataFrame(results)
    Path(output_csv).parent.mkdir(parents=True, exist_ok=True)
    df[["filename", "transcript"]].to_csv(output_csv, index=False, encoding="utf-8-sig")

    total_elapsed = time.time() - total_start
    print(f"\n✅ {len(results)} file(s) processed in {format_time(total_elapsed)}")
    print(f"   Output → {output_csv}")
    return df


print("✅ CT2 inference engine ready.")

# RUN INFERENCE

In [ ]:
inference_start = time.time()
print("=" * 65)
print("🚀  CT2 INFERENCE STARTED")
print(f"    {datetime.now(ZoneInfo('Asia/Dhaka')).strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 65)

df_results = run_ct2_inference(AUDIO_BASE_PATH, OUTPUT_CSV)

total_time = time.time() - inference_start
print("\n" + "=" * 65)
print("🏁  CT2 INFERENCE COMPLETE")
print(f"    Total time : {format_time(total_time)}")
print("=" * 65)

# RESULTS PREVIEW

In [ ]:
if not df_results.empty:
    print(f"\n{'FILE':<40} {'CHARS':>6}  PREVIEW")
    print("─" * 90)
    for _, row in df_results.iterrows():
        preview = str(row["transcript"])[:80].replace("\n", " ")
        print(f"  {row['filename']:<38} {len(str(row['transcript'])):>6}  {preview}…")
    print(f"\n✨ Results saved → {OUTPUT_CSV}")
else:
    print("⚠️  No results to display.")